# W12C2 Lab: Retrieval-Augmented Generation

Run every cell from the top. **Everything already works.**

Uses MiniLM for retrieval (cached from Week 7) and the local model for
generation, with a fallback if Ollama is not running.

Today you will:

1. Build a RAG pipeline end to end with real embeddings.
2. Watch retrieval fail and see the answer fail with it.
3. Poison the knowledge base and see the model repeat the lie.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, logging
from sklearn.metrics.pairwise import cosine_similarity

logging.set_verbosity_error()
enc_tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
enc = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
enc.eval()

KNOWLEDGE = [
    "The library is open from 8am to midnight on weekdays.",
    "The library closes at 6pm on Saturdays and Sundays.",
    "Students may borrow up to 20 books at a time.",
    "Overdue books are fined 25 cents per day, capped at 10 dollars.",
    "The cafe on the ground floor serves coffee until 4pm.",
    "Printing costs 8 cents per black and white page.",
    "Group study rooms can be booked online for up to 3 hours.",
    "The archive on floor 4 requires an appointment.",
]
print(f"{len(KNOWLEDGE)} facts in the knowledge base")

In [ ]:
# GIVEN. Embed everything once, then retrieve by cosine similarity.
@torch.no_grad()
def embed(texts):
    ids = enc_tok(texts, return_tensors="pt", padding=True, truncation=True)
    return enc(**ids).last_hidden_state.mean(dim=1).numpy()

index = embed(KNOWLEDGE)

def retrieve(question, k=2):
    scores = cosine_similarity(embed([question]), index)[0]
    best = np.argsort(scores)[::-1][:k]
    return [(KNOWLEDGE[i], float(scores[i])) for i in best]

for q in ["When does the library shut on Sunday?", "How much does printing cost?"]:
    print(f"Q: {q}")
    for text, score in retrieve(q):
        print(f"   {score:.3f}  {text}")
    print()

In [ ]:
# GIVEN. Retrieval plus generation: the whole of RAG.
import ollama

def ask_llm(prompt):
    try:
        r = ollama.chat(model="qwen2.5:0.5b",
                        messages=[{"role": "user", "content": prompt}],
                        options={"temperature": 0.0})
        return r["message"]["content"].strip()
    except Exception:
        return "(no local model running; showing the prompt only)"

def rag(question, k=2):
    passages = retrieve(question, k)
    context = "\n".join(f"- {t}" for t, s in passages)
    prompt = (f"Answer using ONLY these facts:\n{context}\n\n"
              f"Question: {question}\nAnswer in one short sentence.")
    return ask_llm(prompt), passages

answer, used = rag("When does the library shut on Sunday?")
print("retrieved:")
for t, s in used:
    print(f"   {s:.3f}  {t}")
print("\nanswer:", answer)

In [ ]:
# ================== YOUR TURN 1 ==================
# Ask something the knowledge base cannot answer, for example
# 'What is the wifi password?'
#
# Look at what gets retrieved and what the model then says.
#
# Expected: retrieval ALWAYS returns its two nearest facts, however irrelevant,
#           because nothing in cosine similarity says 'no good match'. The model
#           then answers from them anyway. A similarity floor, below which you
#           refuse to answer, is the missing piece.
# ===============================================
MY_QUESTION = "What is the wifi password?"          # <-- change me

answer, used = rag(MY_QUESTION)
print("retrieved:")
for t, s in used:
    print(f"   {s:.3f}  {t}")
print("\nanswer:", answer)
print()
print("best similarity:", round(used[0][1], 3), "<- is that high enough to trust?")

In [ ]:
# ================== YOUR TURN 2 ==================
# Add the missing piece: refuse to answer when nothing is close enough.
#
# Set THRESHOLD and re-run over all four questions.
#
# Expected: the three real questions score 0.72 to 0.78 and the wifi one scores
#           0.085, so anything from about 0.2 to 0.7 separates them cleanly here. On
#           a real corpus the gap is far narrower and the threshold is the whole
#           design: too high and you refuse questions you could have answered, too
#           low and you answer confidently from irrelevant context.
# ===============================================
THRESHOLD = 0.0          # <-- try 0.3, then 0.5, then 0.7

QUESTIONS = ["When does the library shut on Sunday?",
             "How much does printing cost?",
             "How many books can I borrow?",
             "What is the wifi password?"]

for q in QUESTIONS:
    passages = retrieve(q, k=2)
    best = passages[0][1]
    if best < THRESHOLD:
        print(f"   REFUSED  ({best:.3f})  {q}")
    else:
        print(f"   answered ({best:.3f})  {q}")

## Part 3. Poison the well

RAG grounds the model in your documents. That is its strength and its
weakness: whoever controls the documents controls the answer.

In [ ]:
# GIVEN. Add one false fact and ask again.
POISONED = KNOWLEDGE + ["The library closes at 2pm on Sundays for cleaning."]
poisoned_index = embed(POISONED)

def rag_poisoned(question, k=2):
    scores = cosine_similarity(embed([question]), poisoned_index)[0]
    best = np.argsort(scores)[::-1][:k]
    passages = [(POISONED[i], float(scores[i])) for i in best]
    context = "\n".join(f"- {t}" for t, s in passages)
    return ask_llm(f"Answer using ONLY these facts:\n{context}\n\n"
                   f"Question: {question}\nAnswer in one short sentence."), passages

q = "When does the library shut on Sunday?"
answer, used = rag_poisoned(q)
print("retrieved:")
for t, s in used:
    print(f"   {s:.3f}  {t}")
print("\nanswer:", answer)
print()
print("The true fact says 6pm. The planted one says 2pm. Both were retrieved.")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Cosine similarity always returns a nearest neighbour. It has no concept of
#   "nothing here is relevant", so an unanswerable question retrieves the least
#   irrelevant fact and the model answers from it. Every hallucination in a RAG
#   system starts here, in retrieval, not in the model.
#
# YOUR TURN 2
#   The gap here is huge (0.72 to 0.78 against 0.085) because the knowledge
#   base is tiny and the wifi question shares nothing with it. On a real
#   corpus the gap narrows to a few hundredths. There is no universally right
#   value: you tune it on questions you already know the answers to, which is
#   why the evaluation set has to exist BEFORE the system does.
#
# YOUR TURN 3 (part 3, nothing to edit)
#   The model repeats whichever retrieved fact it likes, and it has no way to
#   know one of them was planted. Defences: control who can write to the index,
#   show sources so a human can check, and prefer facts that several documents
#   agree on.